# 1. Load the dataset

In [16]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from IPython.display import Markdown
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import re

In [17]:
file_path = 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf'
loader = PyPDFLoader(file_path)
document = loader.load()

In [18]:
document[0].metadata

{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)',
 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)',
 'creationdate': '2017-10-31T15:40:29+00:00',
 'author': 'Aurélien Géron',
 'moddate': '2017-10-31T12:10:13-04:00',
 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow',
 'trapped': '/False',
 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf',
 'total_pages': 570,
 'page': 0,
 'page_label': 'Cover'}

In [19]:
print(f'Document has : {len(document)} pages ')
print(f'type of whole document : {type(document)}')
print(f'type of one page: {type(document[0])}')
print(f'meta data of first page: {document[0].metadata}')

Document has : 570 pages 
type of whole document : <class 'list'>
type of one page: <class 'langchain_core.documents.base.Document'>
meta data of first page: {'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2017-10-31T15:40:29+00:00', 'author': 'Aurélien Géron', 'moddate': '2017-10-31T12:10:13-04:00', 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow', 'trapped': '/False', 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf', 'total_pages': 570, 'page': 0, 'page_label': 'Cover'}


## 1.2 split the dataset 

In [20]:
chapter_starts = [
    (24, 'Chapter 1. The Machine Learning Landscape'),
    (54, 'Chapter 2. End-to-End Machine Learning Project'),
    (102, 'Chapter 3. Classification'),
    (128, 'Chapter 4. Training Models'),
    (168, 'Chapter 5. Support Vector Machines'),
    (190, 'Chapter 6. Decision Trees'),
    (204, 'Chapter 7. Ensemble Learning and Random Forests'),
    (228, 'Chapter 8. Dimensionality Reduction'),
    (252, 'Chapter 9. Up and Running with TensorFlow'),
    (276, 'Chapter 10. Introduction to Artificial Neural Networks'),
    (298, 'Chapter 11. Training Deep Neural Nets'),
    (338, 'Chapter 12. Distributing TensorFlow Across Devices and Servers'),
    (378, 'Chapter 13. Convolutional Neural Networks'),
    (404, 'Chapter 14. Recurrent Neural Networks'),
    (438, 'Chapter 15. Autoencoders'),
    (498, 'Appendix A. Exercise Solutions'),
    (524, 'Appendix B. Machine Learning Project Checklist'),
    (530, 'Appendix C. SVM Dual Problem'),
    (534, 'Appendix D. Autodiff'),
    (542, 'Appendix E. Other Popular ANN Architectures'),
]

def get_chapter(page_number):
    chapter = 'Front Matter'
    for start_page, title in chapter_starts:
        if page_number >= start_page:
            chapter = title
        else:
            break
    return chapter

filter_doc = []
for page in document:
    chapter = get_chapter(page.metadata['page'])
    # add chapter info to metadata
    page_label = page.metadata.get('page_label', page.metadata['page'] + 1)
    page.metadata['chapter'] = chapter
    page.metadata['metadata_label'] = f"{chapter} | page {page_label}"
    filter_doc.append(page)


In [21]:
len(filter_doc)
filter_doc[120].metadata

{'producer': 'Antenna House PDF Output Library 6.2.609 (Linux64)',
 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)',
 'creationdate': '2017-10-31T15:40:29+00:00',
 'author': 'Aurélien Géron',
 'moddate': '2017-10-31T12:10:13-04:00',
 'title': 'Hands-On Machine Learning with Scikit-Learn and TensorFlow',
 'trapped': '/False',
 'source': 'data/Hands_On_Machine_Learning_with_Scikit_Learn_and_TensorFlow.pdf',
 'total_pages': 570,
 'page': 120,
 'page_label': '99',
 'chapter': 'Chapter 3. Classification',
 'metadata_label': 'Chapter 3. Classification | page 99'}

# 2. Chunking the dataset into smaller pieces

In [22]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 800,
    chunk_overlap = 150,
    separators=["\n\n", "\n", ". ", " ", ""],
    add_start_index=True
)
chunks = text_splitter.split_documents(filter_doc)


In [23]:
def clean_text(text):
    clean_text= text.encode("utf-8", "ignore").decode("utf-8")
    clean_text = re.sub(r"\s+", " ", clean_text).strip()
    return clean_text

for chunk in chunks:
    chunk.page_content = clean_text(chunk.page_content)

In [24]:
print(f'the length of chunks {len(chunks)}')
print(f'the type of chunks : {type(chunks)}')
print(f'the type of one chunk: {type(chunks[0])}')
print(f'there are total {sum([len(chunk.page_content.lower()) for chunk in chunks])} characters in all chunks')

the length of chunks 1776
the type of chunks : <class 'list'>
the type of one chunk: <class 'langchain_core.documents.base.Document'>
there are total 1154699 characters in all chunks


# 3. Generate embeddings for each chunk

In [25]:
load_dotenv()
api_key = os.getenv('OPENAI_API_KEY')

In [26]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(
    model='text-embedding-3-small'
)

# 4. store the embeddings in a vector database

In [27]:
# Create an FAISS vectore store
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)


## 4.1 test the vector database by querying with a sample question

In [28]:
question = 'what is random forest'
retrieved_docs  = vector_store.similarity_search(query=question,k=10)
retriever_simple = vector_store.as_retriever(
    search_type = 'mmr',
    search_kwargs={"k": 5, "fetch_k": 20}
    )


# 5. Initialize the LLM and set up the retrieval-based question-answering system

## 5.1 Design and implement Multi-stage Retrieval and Re-ranking (MMR) strategy

In [29]:
# Multi-stage Retrieval + rerank
def retrieve_candidates(query,k):

    return vector_store.similarity_search(query,k=20)


rerank_prompt = ChatPromptTemplate.from_template(
    """
    You are a ranking assistant.

    Given a question and a list of documents, rank the documents by relevance.
    Return ONLY a valid JSON list of indices.
    Do NOT use markdown.

    Example:
    [0, 3, 5, 2, 1]

    Question:
    {question}

    Documents:
    {documents}

    Return the indices of the top {top_k} most relevant documents in order.
    """)

rerank_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


In [30]:
import json
def parse_indices(text):
    import json, re

    # 1. remove markdown
    text = re.sub(r"```json|```", "", text).strip()

    # 2. load json
    try:
        return json.loads(text)
    except:
        # 3. fallback
        numbers = re.findall(r"\d+", text)
        return [int(n) for n in numbers]

In [31]:
def rerank_docs(documents,query,top_k):
    format_text  = "\n\n" .join(f"[{i}]\n{doc.metadata.get('metadata_label', 'Unknown source')}\n{doc.page_content}" for i, doc in enumerate(documents))
    response  = rerank_llm.invoke(
        rerank_prompt.format(
            question = query,
            documents = format_text,
            top_k = top_k
        )
    )
    indices = parse_indices(response.content)
    return [documents[i] for i in indices]


In [32]:
result_docs = rerank_docs(retrieved_docs,question,5)

In [34]:
def multi_stage_retrieval(query):
    # Stage 1: recall
    candidates = retrieve_candidates(query, k=20)

    # Stage 2: rerank
    top_docs = rerank_docs(candidates,query,top_k=5)

    return top_docs

In [35]:
# prompt design for LLM to answer
# citation and source
# hallucination
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
        You are a machine learning expert and tutor.

        Your task is to answer the question using ONLY the provided context.

        Guidelines:
        - Use ONLY the information from the context
        - Do NOT use your own knowledge
        - If the answer is not in the context, say: "I don't know based on the provided document"
        - Explain concepts simply
        - Provide examples only if they are supported by the context
        - ALWAYS cite your sources using this format:
          (Source: Chapter X | page Y)
        - If multiple sources are used, cite all of them
        - Be clear and concise

        """
    ),
    (
        "human",
        """
        Context:
        {context}

        Question:
        {question}

        Answer:
        """
    ),
])

In [36]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-4o-mini")

retriever = RunnableLambda(multi_stage_retrieval)

def format_docs(docs):
    return "\n\n".join(
        f"Source:[{doc.metadata.get('metadata_label', 'Unknown source')}]\n Content:{doc.page_content}" for doc in docs
    )


rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)


In [37]:
response = rag_chain.invoke("how to solve poor quality data?")
display(Markdown(response))


To solve poor-quality data, you can take several steps:

1. **Data Cleaning**: Fix or remove outliers whenever possible, and fill in missing values using methods such as zero, mean, or median. Alternatively, you can drop rows or columns with missing data (Source: Appendix B | page 505).

2. **Manual Correction**: Discard instances that are clearly outliers or try to fix them manually (Source: Chapter 1 | page 25).

3. **Monitoring Input Data Quality**: Evaluate the system’s input data quality regularly to catch any performance degradation early. This is especially important in online learning systems (Source: Chapter 2 | page 78).

4. **Feature Selection**: Drop attributes that provide no useful information for the task to improve data quality (Source: Appendix B | page 505).

By implementing these practices, you can significantly enhance the quality of your training data, leading to better performance of your machine learning system.

# 6. Evalaution of the system

## 6.1 Build a small benchmark set
Use a fixed set of questions so you can compare retrieval and answer quality after every change.

In [44]:
eval_set = [
    {
        "question": "What is unsupervised learning?",
        "expected_chapter": "Chapter 1",
        "expected_terms": ["unlabeled", "patterns", "without a teacher"],
    },
    {
        "question": "What are the main steps in a machine learning project?",
        "expected_chapter": "Chapter 2",
        "expected_terms": ["frame the problem", "get the data", "select and train a model"],
    },
    {
        "question": "What is the ROC curve used for?",
        "expected_chapter": "Chapter 3",
        "expected_terms": ["classifier", "tradeoff", "true positive"],
    },
    {
        "question": "What is the difference between batch gradient descent and stochastic gradient descent?",
        "expected_chapter": "Chapter 4",
        "expected_terms": ["entire training set", "single instance", "gradient descent"],
    },
    {
        "question": "What is a support vector machine?",
        "expected_chapter": "Chapter 5",
        "expected_terms": ["svm", "classification", "regression"],
    },
    {
        "question": "How do decision trees make predictions?",
        "expected_chapter": "Chapter 6",
        "expected_terms": ["root node", "leaf", "split"],
    },
    {
        "question": "What is a random forest?",
        "expected_chapter": "Chapter 7",
        "expected_terms": ["decision trees", "ensemble", "random forest"],
    },
    {
        "question": "Why do we use dimensionality reduction?",
        "expected_chapter": "Chapter 8",
        "expected_terms": ["curse of dimensionality", "speed", "memory"],
    },
    {
        "question": "What is TensorFlow?",
        "expected_chapter": "Chapter 9",
        "expected_terms": ["open source", "numerical computation", "machine learning"],
    },
    {
        "question": "What is an autoencoder?",
        "expected_chapter": "Chapter 15",
        "expected_terms": ["neural network", "representation", "coding"],
    },
]

len(eval_set)


10

## 6.2 Run retrieval and answer evaluation
For each question, measure whether retrieval found the expected chapter and whether the final answer contains the expected chapter citation and key terms.

In [45]:
from pprint import pprint

def extract_source_labels(docs):
    return [doc.metadata.get("metadata_label", "Unknown source") for doc in docs]

def evaluate_one_question(item):
    question = item["question"]
    expected_chapter = item["expected_chapter"]
    expected_terms = item.get("expected_terms", [])
    docs = multi_stage_retrieval(question)
    retrieved_sources = extract_source_labels(docs)
    answer = rag_chain.invoke(question)

    retrieval_hit = any(expected_chapter in source for source in retrieved_sources)
    citation_hit = expected_chapter in answer
    matched_terms = [term for term in expected_terms if term.lower() in answer.lower()]

    return {
        "question": question,
        "expected_chapter": expected_chapter,
        "retrieved_sources": retrieved_sources,
        "retrieval_hit": retrieval_hit,
        "answer": answer,
        "citation_hit": citation_hit,
        "matched_terms": matched_terms,
        "term_hit_rate": round(len(matched_terms) / max(len(expected_terms), 1), 2),
    }

eval_results = [evaluate_one_question(item) for item in eval_set]
len(eval_results)


KeyboardInterrupt: 

In [32]:
total_questions = len(eval_results)
retrieval_hit_rate = sum(item["retrieval_hit"] for item in eval_results) / total_questions
citation_hit_rate = sum(item["citation_hit"] for item in eval_results) / total_questions
avg_term_hit_rate = sum(item["term_hit_rate"] for item in eval_results) / total_questions

summary = {
    "total_questions": total_questions,
    "retrieval_hit_rate@top_docs": round(retrieval_hit_rate, 2),
    "citation_hit_rate": round(citation_hit_rate, 2),
    "average_term_hit_rate": round(avg_term_hit_rate, 2),
}

summary


{'total_questions': 10,
 'retrieval_hit_rate@top_docs': 1.0,
 'citation_hit_rate': 0.9,
 'average_term_hit_rate': 0.83}

In [33]:
for item in eval_results:
    pprint({
        "question": item["question"],
        "expected_chapter": item["expected_chapter"],
        "retrieval_hit": item["retrieval_hit"],
        "citation_hit": item["citation_hit"],
        "matched_terms": item["matched_terms"],
        "retrieved_sources": item["retrieved_sources"][:3],
    })
    print("-" * 80)


{'citation_hit': True,
 'expected_chapter': 'Chapter 1',
 'matched_terms': ['unlabeled', 'patterns', 'without a teacher'],
 'question': 'What is unsupervised learning?',
 'retrieval_hit': True,
 'retrieved_sources': ['Chapter 1. The Machine Learning Landscape | page 10',
                       'Chapter 1. The Machine Learning Landscape | page 12',
                       'Chapter 1. The Machine Learning Landscape | page 11']}
--------------------------------------------------------------------------------
{'citation_hit': False,
 'expected_chapter': 'Chapter 2',
 'matched_terms': ['frame the problem', 'get the data'],
 'question': 'What are the main steps in a machine learning project?',
 'retrieval_hit': True,
 'retrieved_sources': ['Appendix B. Machine Learning Project Checklist | page '
                       '503',
                       'Chapter 2. End-to-End Machine Learning Project | page '
                       '33',
                       'Chapter 2. End-to-End Machine Learnin

## 6.3 How to use these results
- If `retrieval_hit_rate` is low, improve chunking, metadata, or retrieval settings first.
- If retrieval is good but `citation_hit_rate` or `term_hit_rate` is low, improve the answer prompt or reranking.
- Keep this benchmark fixed and rerun it after every major RAG change.

# 7. Improved Evaluation 

## 7.1 LLM-as-a-Judge
Use a second LLM to review the answer against the retrieved context and return structured evaluation scores.

In [74]:
judge_prompt = ChatPromptTemplate.from_template(
    """
    You are an evaluator for a retrieval-augmented question answering system.

    Score the answer using ONLY the question, retrieved context, expected chapter, expected terms, and the answer itself.
    Do not use outside knowledge.

    Return ONLY valid JSON.
    Use these exact keys:
    - correctness: integer from 1 to 5
    - groundedness: integer from 1 to 5
    - citation_quality: integer from 1 to 5
    - hallucination_risk: integer from 1 to 5
    - verdict: string, either pass or fail
    - reason: short explanation

    Scoring guide:
    - correctness: Is the answer factually correct based on the context?
    - groundedness: Is the answer supported by the retrieved context?
    - citation_quality: Are the citations present and aligned with the answer?
    - hallucination_risk: 1 means low risk, 5 means high risk.

    Question:
    {question}

    Expected chapter:
    {expected_chapter}

    Expected terms:
    {expected_terms}

    Retrieved context:
    {context}

    Answer:
    {answer}
    """
)

judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def parse_judge_json(text):
    cleaned = re.sub(r"```json|```", "", text).strip()
    return json.loads(cleaned)

def judge_one_question(item):
    question = item["question"]
    expected_chapter = item["expected_chapter"]
    expected_terms = item.get("expected_terms", [])

    docs = multi_stage_retrieval(question)
    context = format_docs(docs)
    answer = rag_chain.invoke(question)

    response = judge_llm.invoke(
        judge_prompt.format(
            question=question,
            expected_chapter=expected_chapter,
            expected_terms=", ".join(expected_terms),
            context=context,
            answer=answer,
        )
    )

    scores = parse_judge_json(response.content)

    return {
        "question": question,
        "expected_chapter": expected_chapter,
        "answer": answer,
        "judge": scores,
    }


In [ ]:
expected_terms = eval_set[0].get('expected_terms')

['unlabeled', 'patterns', 'without a teacher']

In [ ]:
question = eval_set[0]['question']
expected_chapter = eval_set[0]['expected_chapter']
Markdown(judge_llm.invoke(
    judge_prompt.format(
        question=question,
            expected_chapter=expected_chapter,
            expected_terms=", ".join(expected_terms),
            context=context,
            answer=answer,
    )
    )

Unsupervised learning is a type of machine learning where the model is trained on data that does not have labeled outputs. In other words, the algorithm is provided with input data without any corresponding target values or categories. The goal of unsupervised learning is to identify patterns, structures, or relationships within the data.

Key characteristics of unsupervised learning include:

1. **No Labeled Data**: Unlike supervised learning, where the model learns from labeled examples (input-output pairs), unsupervised learning works with data that has no labels.

2. **Pattern Discovery**: The primary objective is to discover hidden patterns or intrinsic structures in the data. This can include clustering similar data points, reducing dimensionality, or identifying anomalies.

3. **Common Techniques**:
   - **Clustering**: Grouping similar data points together. Examples include K-means clustering, hierarchical clustering, and DBSCAN.
   - **Dimensionality Reduction**: Reducing the number of features in the dataset while preserving important information. Techniques include Principal Component Analysis (PCA) and t-Distributed Stochastic Neighbor Embedding (t-SNE).
   - **Association Rule Learning**: Discovering interesting relationships between variables in large datasets, often used in market basket analysis.

4. **Applications**: Unsupervised learning is used in various applications, such as customer segmentation, anomaly detection, recommendation systems, and exploratory data analysis.

Overall, unsupervised learning is a powerful approach for analyzing and interpreting complex datasets where labeled data is scarce or unavailable.

In [75]:
judge_result = judge_one_question(eval_set[0])
judge_result

{'question': 'What is unsupervised learning?',
 'expected_chapter': 'Chapter 1',
 'answer': 'Unsupervised learning is a type of machine learning where the training data is unlabeled, meaning there is no explicit teacher guiding the system. The algorithm attempts to learn patterns and structures from the data independently (Source: Chapter 1 | page 10). Important tasks in unsupervised learning include clustering (such as k-Means and Hierarchical Cluster Analysis), visualization and dimensionality reduction (like Principal Component Analysis), and association rule learning (for example, the Apriori algorithm) (Source: Chapter 1 | page 10 and page 12). Additionally, unsupervised learning can be used for anomaly detection, which involves identifying unusual instances, such as detecting fraudulent credit card transactions (Source: Chapter 1 | page 12).',
 'judge': {'correctness': 5,
  'groundedness': 5,
  'citation_quality': 5,
  'hallucination_risk': 1,
  'verdict': 'pass',
  'reason': 'Th

In [76]:
judge_results = [judge_one_question(item) for item in eval_set]

judge_summary = {
    "avg_correctness": round(sum(item["judge"]["correctness"] for item in judge_results) / len(judge_results), 2),
    "avg_groundedness": round(sum(item["judge"]["groundedness"] for item in judge_results) / len(judge_results), 2),
    "avg_citation_quality": round(sum(item["judge"]["citation_quality"] for item in judge_results) / len(judge_results), 2),
    "avg_hallucination_risk": round(sum(item["judge"]["hallucination_risk"] for item in judge_results) / len(judge_results), 2),
    "pass_rate": round(sum(item["judge"]["verdict"] == "pass" for item in judge_results) / len(judge_results), 2),
}

judge_summary

{'avg_correctness': 4.8,
 'avg_groundedness': 4.9,
 'avg_citation_quality': 4.8,
 'avg_hallucination_risk': 1.1,
 'pass_rate': 0.9}